In [22]:
# ==============================
# INSTALL REQUIRED LIBRARIES
# ==============================

!pip install -q wandb
!pip install -q librosa
!pip install -q tqdm
!pip install transformers

In [23]:
# =========================================
# IMPORTS + DEVICE SETUP + RANDOM SEED
# =========================================

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import ASTForAudioClassification

import librosa
import librosa.display

from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split

from tqdm import tqdm
import wandb

import warnings
warnings.filterwarnings("ignore")

2026-02-17 07:41:26.789340: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771314086.991729      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771314087.055533      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771314087.524106      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771314087.524135      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771314087.524138      55 computation_placer.cc:177] computation placer alr

In [3]:
# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
# -----------------------------
# Seed for reproducibility
# -----------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [30]:
class CFG:
    
    DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
    GENRE_STEMS = os.path.join(DATA_ROOT, "genres_stems")
    MASHUP_DIR = os.path.join(DATA_ROOT, "mashups")
    TEST_CSV = os.path.join(DATA_ROOT, "test.csv")
    SAMPLE_SUB = os.path.join(DATA_ROOT, "sample_submission.csv")
    
    # Audio
    SAMPLE_RATE = 16000
    DURATION = 30
    MAX_LENGTH = SAMPLE_RATE * DURATION
    N_MELS = 128
    
    # Training
    BATCH_SIZE = 8
    EPOCHS = 8
    LR = 2e-5
    WEIGHT_DECAY = 0.01
    
    # Misc
    NUM_CLASSES = 10
    SEED = 42
    MODEL_SAVE_PATH = "ast_model.pth"
    
    TRAIN_MODE = True

In [6]:
# =========================================
# LABEL MAPPING + TRAIN/VAL SPLIT
# =========================================

GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]

label2idx = {genre: idx for idx, genre in enumerate(GENRES)}
idx2label = {idx: genre for genre, idx in label2idx.items()}

print("Label mapping:", label2idx)


# Collect all song folders (each song contains 4 stems)
all_songs = []

for genre in GENRES:
    genre_path = os.path.join(CFG.GENRE_STEMS, genre)
    song_folders = os.listdir(genre_path)
    
    for song in song_folders:
        song_path = os.path.join(genre_path, song)
        all_songs.append((song_path, label2idx[genre]))

print("Total songs found:", len(all_songs))


# Train/Validation split
train_songs, val_songs = train_test_split(
    all_songs,
    test_size=0.2,
    stratify=[label for _, label in all_songs],
    random_state=CFG.SEED
)

print("Train samples:", len(train_songs))
print("Validation samples:", len(val_songs))

Label mapping: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}
Total songs found: 1000
Train samples: 800
Validation samples: 200


In [7]:
# =========================================
# LOAD ESC-50 NOISE FILE PATHS
# =========================================

ESC_AUDIO_DIR = os.path.join(CFG.DATA_ROOT, "ESC-50-master", "audio")

noise_files = [
    os.path.join(ESC_AUDIO_DIR, f)
    for f in os.listdir(ESC_AUDIO_DIR)
    if f.endswith(".wav")
]

print("Total noise files found:", len(noise_files))

Total noise files found: 2000


In [35]:
# =========================================
# MASHUP DATASET FOR AST (LOG-MEL)
# =========================================

class MashupDataset(Dataset):
    def __init__(self, song_list, train=True):
        self.song_list = song_list
        self.train = train
        self.sample_rate = CFG.SAMPLE_RATE
        self.max_length = CFG.SAMPLE_RATE * CFG.DURATION

    def load_audio(self, path):
        audio, _ = librosa.load(path, sr=self.sample_rate)

        # Trim or pad to fixed length
        if len(audio) > self.max_length:
            audio = audio[:self.max_length]
        else:
            pad_length = self.max_length - len(audio)
            audio = np.pad(audio, (0, pad_length))

        return audio

    def add_noise(self, audio):
        noise_path = random.choice(noise_files)
        noise, _ = librosa.load(noise_path, sr=self.sample_rate)

        # Pad/trim noise to match length
        if len(noise) > self.max_length:
            noise = noise[:self.max_length]
        else:
            pad_length = self.max_length - len(noise)
            noise = np.pad(noise, (0, pad_length))

        alpha = random.uniform(0.01, 0.05)
        return audio + alpha * noise

    def __len__(self):
        return len(self.song_list)

    def __getitem__(self, idx):
        song_path, label = self.song_list[idx]

        # Load stems
        drums = self.load_audio(os.path.join(song_path, "drums.wav"))
        vocals = self.load_audio(os.path.join(song_path, "vocals.wav"))
        bass = self.load_audio(os.path.join(song_path, "bass.wav"))
        other = self.load_audio(os.path.join(song_path, "other.wav"))

        # Stem gain augmentation
        if self.train:
            drums *= random.uniform(0.7, 1.3)
            vocals *= random.uniform(0.7, 1.3)
            bass *= random.uniform(0.7, 1.3)
            other *= random.uniform(0.7, 1.3)

        # Mix stems
        mixed_audio = (drums + vocals + bass + other) / 4.0

        # Add noise with probability
        if self.train and random.random() < 0.7:
            mixed_audio = self.add_noise(mixed_audio)

        # Prevent clipping distortion
        mixed_audio = np.clip(mixed_audio, -1.0, 1.0)

        # ===============================
        # Log-Mel Spectrogram
        # ===============================
        mel = librosa.feature.melspectrogram(
            y=mixed_audio,
            sr=self.sample_rate,
            n_mels=CFG.N_MELS,
            fmax=8000
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)

        # Normalize
        mel_db = (mel_db - np.mean(mel_db)) / (np.std(mel_db) + 1e-6)

        mel_db = torch.tensor(mel_db, dtype=torch.float32)
        target_frames = 1024  # Safe fixed length

        if mel_db.shape[1] > target_frames:
            mel_db = mel_db[:, :target_frames]
        else:
            pad_width = target_frames - mel_db.shape[1]
            mel_db = torch.nn.functional.pad(mel_db, (0, pad_width))


        # Output shape: (128, T)
        return mel_db, label

In [36]:
dataset_test = MashupDataset(train_songs)
sample_mel, sample_label = dataset_test[0]

print("Log-Mel shape:", sample_mel.shape)
print("Label:", sample_label)

Log-Mel shape: torch.Size([128, 1024])
Label: 5


In [37]:
train_dataset = MashupDataset(train_songs, train=True)
val_dataset = MashupDataset(val_songs, train=False)

train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [11]:
# # =========================================
# # CNN MODEL (MFCC INPUT)
# # =========================================

# class CNN_MFCC(nn.Module):
#     def __init__(self, num_classes=CFG.NUM_CLASSES):
#         super(CNN_MFCC, self).__init__()
        
#         self.features = nn.Sequential(
#             nn.Conv2d(1, 16, kernel_size=3, padding=1),
#             nn.BatchNorm2d(16),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
            
#             nn.Conv2d(16, 32, kernel_size=3, padding=1),
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
            
#             nn.Conv2d(32, 64, kernel_size=3, padding=1),
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#         )
        
#         # We will compute this dynamically
#         self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(64, 128),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(128, num_classes)
#         )
        
#     def forward(self, x):
#         x = self.features(x)
#         x = self.global_pool(x)
#         x = self.classifier(x)
#         return x


# # Initialize model
# model = CNN_MFCC().to(device)

# print(model)

CNN_MFCC(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (global_pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=64, out_featur

In [13]:
# # =========================================
# # LSTM MODEL (MFCC SEQUENCE)
# # =========================================

# class LSTM_MFCC(nn.Module):
#     def __init__(self, input_size=CFG.N_MFCC, hidden_size=128, num_layers=2, num_classes=CFG.NUM_CLASSES):
#         super(LSTM_MFCC, self).__init__()
        
#         self.lstm = nn.LSTM(
#             input_size=input_size,
#             hidden_size=hidden_size,
#             num_layers=num_layers,
#             batch_first=True,
#             bidirectional=True
#         )
        
#         self.dropout = nn.Dropout(0.3)
        
#         self.fc = nn.Linear(hidden_size * 2, num_classes)
        
#     def forward(self, x):
#         # x shape: (batch, 1, 40, T)
        
#         x = x.squeeze(1)          # (batch, 40, T)
#         x = x.permute(0, 2, 1)    # (batch, T, 40)
        
#         output, _ = self.lstm(x)
        
#         # Take last time step
#         last_output = output[:, -1, :]
        
#         out = self.dropout(last_output)
#         out = self.fc(out)
        
#         return out

In [14]:
# model = LSTM_MFCC().to(device)
# print(model)

LSTM_MFCC(
  (lstm): LSTM(40, 128, num_layers=2, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=10, bias=True)
)


In [39]:
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=CFG.NUM_CLASSES,
    ignore_mismatched_sizes=True
)

model = model.to(device)

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("DLGENAI_WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = secret_value_0
wandb.login()

wandb: Currently logged in as: sharibahmad (sharibahmad-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [42]:
# =========================================
# LOSS, OPTIMIZER, WANDB SETUP (SAFE MODE)
# =========================================

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.LR,
    weight_decay=0.01
)


import wandb

if CFG.TRAIN_MODE:
    wandb.init(
        project="24f2001786-t12026",
        name="ast_finetuned_model",
        config={
            "epochs": CFG.EPOCHS,
            "batch_size": CFG.BATCH_SIZE,
            "lr": CFG.LR,
            "n_mfcc": CFG.N_MELS
        },
        reinit=True
    )

print("WandB safely initialized.")

WandB safely initialized.


In [44]:
x, y = next(iter(train_loader))

print("Before permute:", x.shape)

x = x.permute(0, 2, 1)

print("After permute:", x.shape)

Before permute: torch.Size([8, 128, 1024])
After permute: torch.Size([8, 1024, 128])


In [45]:
# =========================================
# TRAINING + VALIDATION LOOP (AST VERSION)
# =========================================

from sklearn.metrics import f1_score, accuracy_score

best_f1 = 0.0

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    loop = tqdm(loader, desc="Training", leave=False)
    
    for inputs, labels in loop:
        inputs = inputs.to(device)
        inputs = inputs.permute(0, 2, 1)  # (B, 1024, 128)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(inputs).logits
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # 🔥 Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
        
        loop.set_postfix(loss=loss.item())
    
    epoch_loss = total_loss / len(loader)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")
    epoch_acc = accuracy_score(all_labels, all_preds)
    
    return epoch_loss, epoch_f1, epoch_acc


def validate(model, loader):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    loop = tqdm(loader, desc="Validation", leave=False)
    
    with torch.no_grad():
        for inputs, labels in loop:
            inputs = inputs.to(device)
            inputs = inputs.permute(0, 2, 1)  # (B, 1024, 128)
            labels = labels.to(device)
            
            outputs = model(inputs).logits
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
            
            loop.set_postfix(loss=loss.item())
    
    epoch_loss = total_loss / len(loader)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")
    epoch_acc = accuracy_score(all_labels, all_preds)
    
    return epoch_loss, epoch_f1, epoch_acc


# =========================
# TRAINING START
# =========================

if CFG.TRAIN_MODE:
    
    for epoch in range(CFG.EPOCHS):
        print(f"\nEpoch {epoch+1}/{CFG.EPOCHS}")
        
        train_loss, train_f1, train_acc = train_one_epoch(model, train_loader)
        val_loss, val_f1, val_acc = validate(model, val_loader)
        
        print(f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   F1: {val_f1:.4f} | Val   Acc: {val_acc:.4f}")
        
        # WandB Logging
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_f1": train_f1,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_f1": val_f1,
            "val_acc": val_acc
        })
        
        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), CFG.MODEL_SAVE_PATH)
            wandb.save(CFG.MODEL_SAVE_PATH)
            print("✅ Best model saved!")


Epoch 1/8


Train Loss: 1.3807 | Train F1: 0.5317 | Train Acc: 0.5400
Val   Loss: 0.8610 | Val   F1: 0.7085 | Val   Acc: 0.7250
✅ Best model saved!

Epoch 2/8


Train Loss: 0.6404 | Train F1: 0.7963 | Train Acc: 0.8000
Val   Loss: 0.7293 | Val   F1: 0.7388 | Val   Acc: 0.7400
✅ Best model saved!

Epoch 3/8


Train Loss: 0.3601 | Train F1: 0.8880 | Train Acc: 0.8888
Val   Loss: 0.6266 | Val   F1: 0.7770 | Val   Acc: 0.7750
✅ Best model saved!

Epoch 4/8


Train Loss: 0.1822 | Train F1: 0.9332 | Train Acc: 0.9337
Val   Loss: 0.8427 | Val   F1: 0.7772 | Val   Acc: 0.7700
✅ Best model saved!

Epoch 5/8


Train Loss: 0.1253 | Train F1: 0.9625 | Train Acc: 0.9625
Val   Loss: 0.5941 | Val   F1: 0.8577 | Val   Acc: 0.8550
✅ Best model saved!

Epoch 6/8


Train Loss: 0.0398 | Train F1: 0.9887 | Train Acc: 0.9888
Val   Loss: 0.6912 | Val   F1: 0.8243 | Val   Acc: 0.8250

Epoch 7/8


Train Loss: 0.0245 | Train F1: 0.9937 | Train Acc: 0.9938
Val   Loss: 0.9943 | Val   F1: 0.8005 | Val   Acc: 0.8000

Epoch 8/8


Train Loss: 0.0119 | Train F1: 0.9962 | Train Acc: 0.9962
Val   Loss: 0.6600 | Val   F1: 0.8429 | Val   Acc: 0.8450


In [46]:
# =========================================
# LOAD BEST MODEL FOR INFERENCE
# =========================================

MODEL_PATH = "/kaggle/input/models/genrede/ast/transformers/default/1/ast_model.pth"
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

print("Model loaded successfully.")


Model loaded successfully.


In [47]:
# =========================================
# TEST DATASET (AST - LOG MEL VERSION)
# =========================================

class TestDataset(Dataset):
    def __init__(self, test_csv_path, data_root):
        self.test_df = pd.read_csv(test_csv_path)
        self.data_root = data_root
        self.sample_rate = CFG.SAMPLE_RATE
        self.max_length = CFG.SAMPLE_RATE * CFG.DURATION
        self.target_frames = 1024  # Must match training

    def load_audio(self, path):
        audio, _ = librosa.load(path, sr=self.sample_rate)

        if len(audio) > self.max_length:
            audio = audio[:self.max_length]
        else:
            pad_length = self.max_length - len(audio)
            audio = np.pad(audio, (0, pad_length))

        return audio

    def __len__(self):
        return len(self.test_df)

    def __getitem__(self, idx):
        row = self.test_df.iloc[idx]
        file_id = row["id"]
        filename = row["filename"]

        audio_path = os.path.join(CFG.DATA_ROOT, filename)
        audio = self.load_audio(audio_path)

        # Log-Mel Spectrogram
        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=self.sample_rate,
            n_mels=CFG.N_MELS,
            fmax=8000
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)

        # Normalize
        mel_db = (mel_db - np.mean(mel_db)) / (np.std(mel_db) + 1e-6)

        mel_db = torch.tensor(mel_db, dtype=torch.float32)

        # Ensure fixed 1024 frames
        if mel_db.shape[1] > self.target_frames:
            mel_db = mel_db[:, :self.target_frames]
        else:
            pad_width = self.target_frames - mel_db.shape[1]
            mel_db = torch.nn.functional.pad(mel_db, (0, pad_width))

        # Final shape: (128, 1024)
        return mel_db, file_id

In [48]:
test_dataset = TestDataset(CFG.TEST_CSV, CFG.DATA_ROOT)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Test samples:", len(test_dataset))

Test samples: 3020


In [49]:
# =========================================
# AST INFERENCE (CORRECT VERSION)
# =========================================

predictions = []
ids = []

model.eval()

with torch.no_grad():
    for inputs, file_ids in tqdm(test_loader, desc="Inference"):
        inputs = inputs.to(device)
        inputs = inputs.permute(0, 2, 1)  # (B, 1024, 128)
        
        outputs = model(inputs).logits
        preds = torch.argmax(outputs, dim=1)
        
        predictions.extend(preds.cpu().numpy())
        ids.extend([int(i) for i in file_ids])

# Convert indices to genre names
predicted_genres = [idx2label[p] for p in predictions]

submission = pd.DataFrame({
    "id": ids,
    "genre": predicted_genres
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Saved at:", os.path.exists("/kaggle/working/submission.csv"))
print("Submission file created successfully!")

submission.head()

Inference: 100%|██████████| 378/378 [04:33<00:00,  1.38it/s]

Saved at: True
Submission file created successfully!


,id,genre
0,1,pop
1,2,jazz
2,3,disco
3,4,metal
4,5,rock
